# ICE for LLMs — Replicating Hou et al. (ICML 2024)

Comparison of the **Input Clarification Ensembling (ICE)** method on NLP tasks,
following the experimental setup from:

> Hou et al., *"Decomposing Uncertainty for Large Language Models through
> Input Clarification Ensembling"* (arXiv:2311.08718v2, ICML 2024)

**Datasets**: Natural Questions (NQ), AmbigQA, GSM8K

**Pipeline** (5 steps):
1. **Generate clarifications** — LLM rephrases each question into multiple variants
2. **Forward pass** — sample N=10 answers per clarification (temperature sampling)
3. **Answer extraction** — LLM-based semantic answer clustering
4. **Uncertainty quantification** — entropy decomposition (total = model + data)
5. **Evaluation** — AUROC / best-F1 for mistake detection & ambiguity detection

**Uncertainty decomposition (ICE)**:
- **Total** H = entropy of the aggregated answer distribution
- **Model uncertainty** = avg conditional entropy (within-clarification variance)
- **Data uncertainty** = MI = H_total − model_unc (between-clarification disagreement)

For **mistake detection** (NQ, GSM8K): high total uncertainty → likely wrong
For **ambiguity detection** (AmbigQA): high data uncertainty → question is ambiguous


In [ ]:
!pip install -q openai>=1.0 datasets scikit-learn jiwer regex tqdm

In [ ]:
import os

# ─── USER CONFIG ───────────────────────────────────────────────
API_KEY        = ""          # paste your OpenAI key here
MODEL_CLARIFY  = "gpt-4o-mini"   # model for clarification generation
MODEL_FORWARD  = "gpt-4o-mini"   # model for answer sampling
MODEL_EXTRACT  = "gpt-4o-mini"   # model for answer extraction
MODEL_EVAL     = "gpt-4o-mini"   # model for NQ correctness scoring

MAX_CLARIFY    = 5           # clarifications per question (paper: 5)
SAMPLE_N       = 10          # answer samples per clarification (paper: 10)
TEMPERATURE    = 0.5         # sampling temperature for forward pass

# Dataset sizes (paper: NQ=200, AmbigQA=200, GSM8K=200)
N_NQ           = 200
N_AMBIGQA      = 200         # 100 ambig + 100 unambig
N_GSM8K        = 200

# Set to True to run the full pipeline (requires API calls).
# Set to False to load saved results from CACHE_DIR.
RUN_PIPELINE   = True

CACHE_DIR      = "ice_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

if not API_KEY:
    from google.colab import userdata
    try:
        API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
assert API_KEY, "Set API_KEY above or store OPENAI_API_KEY in Colab Secrets"


In [ ]:
import json, re, string, copy, time
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI
from tenacity import retry, wait_exponential, stop_after_attempt
import datasets
from sklearn.metrics import roc_auc_score, f1_score
from jiwer import wer
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
client = OpenAI(api_key=API_KEY)


In [ ]:
@retry(wait=wait_exponential(min=1, max=30), stop=stop_after_attempt(6))
def chat_complete(model, messages, temperature=0, max_tokens=512, n=1):
    """Wrapper around OpenAI chat completions with retry."""
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        n=n,
    )
    return [c.message.content for c in resp.choices]

def format_messages(system_prompt, user_prompt):
    msgs = []
    if system_prompt:
        msgs.append({"role": "system", "content": system_prompt})
    msgs.append({"role": "user", "content": user_prompt})
    return msgs


In [ ]:
import regex as re_regex

def normalize_answer(s):
    def remove_articles(text):
        return re_regex.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        if 'regard' in text:
            text = re_regex.sub(r'\([^)]*\)', '', text).strip()
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def recursive_normalize(obj):
    if isinstance(obj, list):
        return [recursive_normalize(x) for x in obj]
    elif isinstance(obj, str):
        return normalize_answer(obj)
    else:
        return obj

UNK_WORDS = [
    'unknown', "I don't", "did not", "Not specified", "cannot be determined",
    "No Answer", "No final answer", "I do not", "N/A", "No information",
    "It depends on", "I cannot", "I can't ", "I am unable", "don't know",
    "No answer", "Nobody", "enough information", "specific information",
    "There is no", "No specific", "not provided", "None", "No character",
    "did not", "No output", "Cannot answer", "Unavailable", "TBD",
    "To Be Determined", "No translation", "depends on", "has not",
    'unclear', "confusion", "incorrect", "not aware of", "invalid", 'no one'
]

def check_answers(ans_list):
    unk_lower = [x.lower() for x in UNK_WORDS]
    purified = []
    for ans in ans_list:
        flag = False
        if ans.lower() in unk_lower:
            flag = True
        for uw in UNK_WORDS:
            if uw.lower() in ans.lower():
                flag = True
                break
        purified.append('unknown' if flag else ans)
    return purified

def majority_vote(answers):
    freq = {}
    best_ans, best_freq = None, 0
    for a in answers:
        freq[a] = freq.get(a, 0) + 1
        if freq[a] > best_freq:
            best_ans, best_freq = a, freq[a]
    return best_ans, best_freq

def gsm8k_extract_ans(pred_str):
    preds = re.findall(r"\$?([0-9,]+)\.?\d*%?", str(pred_str))
    if preds:
        p = preds[-1].replace(",", "").replace(" ", "")
        try:
            return int(p)
        except ValueError:
            return -1
    return -1

class FuzzingDict:
    def __init__(self, ans2id):
        self.ans2id = ans2id
        self.id2ans = {v: k for k, v in ans2id.items()}
        self.answer_list = [self.id2ans[i] for i in range(len(ans2id))]

    def __call__(self, key):
        if key in self.ans2id:
            return self.ans2id[key]
        sim = [a for a in self.answer_list if a in key or key in a]
        if len(sim) == 1:
            return self.ans2id[sim[0]]
        if len(sim) == 0:
            return self.ans2id[self.answer_list[0]]
        dists = np.array([wer(key, a) for a in sim])
        return self.ans2id[sim[np.argmin(dists)]]


In [ ]:
def load_nq(n=200):
    ds = datasets.load_dataset('nq_open', cache_dir='dataset_cache', trust_remote_code=True)
    data = [ds['validation'][i] for i in range(min(n, len(ds['validation'])))]
    print(f"NQ: loaded {len(data)} examples")
    return data

def load_ambigqa(n=200):
    ds = datasets.load_dataset('ambig_qa', 'light', cache_dir='dataset_cache', trust_remote_code=True)
    val = ds['validation']
    ambig, unambig = [], []
    for ex in val:
        ann = ex['annotations']
        has_multi = any(t != 'singleAnswer' for t in ann['type'])
        item = {
            'question': ex['question'],
            'answer': [a for sublist in ann['answer'] for a in sublist],
            'label': ann['type'],
        }
        if has_multi and len(ambig) < n // 2:
            ambig.append(item)
        elif not has_multi and len(unambig) < n // 2:
            unambig.append(item)
        if len(ambig) >= n // 2 and len(unambig) >= n // 2:
            break
    data = ambig + unambig
    print(f"AmbigQA: loaded {len(data)} examples ({len(ambig)} ambig, {len(unambig)} unambig)")
    return data

def load_gsm8k(n=200):
    ds = datasets.load_dataset('gsm8k', 'main', cache_dir='dataset_cache', trust_remote_code=True)
    data = [ds['test'][i] for i in range(min(n, len(ds['test'])))]
    print(f"GSM8K: loaded {len(data)} examples")
    return data


## Step 1 — Generate Clarifications

For each question, the LLM generates multiple rephrased versions.
These serve as the "input ensemble" in ICE.

- **NQ / GSM8K**: Rephrase the question (same meaning, different wording)
- **AmbigQA**: Disambiguate the question (resolve potential ambiguities)


In [ ]:
# ── Clarification system prompts (from Hou et al.) ──

CLARIFY_PROMPT_NQ = """In this task, you will receive a single question, and your goal is to generate multiple versions of it that convey the same meaning as the original. Please format your responses as follows:
Rephrase 1: [Your rephrased question]
Rephrase 2: [Another rephrased question]
Rephrase 3: [Yet another rephrased question]
....
Ensure that each rephrased question is distinct from the others.

Here are two examples:
Question: When did the manhattan project began and end?
Rephrase 1: What were the start and end dates of the Manhattan Project?
Rephrase 2: The manhattan project began and ended in ?
Rephrase 3: What were the starting and ending dates of the Manhattan Project?

Question: Who played george washington in the john adams series?
Rephrase 1: In the John Adams series, who portrayed George Washington?
Rephrase 2: In the John Adams series, which actor portrayed George Washington?
Rephrase 3: Who portrayed George Washington in the John Adams series?
"""

CLARIFY_PROMPT_GSM8K = """In this task, you will receive a single question, and your goal is to generate multiple versions of it that convey the same meaning as the original. Please format your responses as follows:
Rephrase 1: [Your rephrased question]
Rephrase 2: [Another rephrased question]
Rephrase 3: [Yet another rephrased question]
....

Here are two examples:
Question: John has 3 boxes.  Each box is 5 inches by 6 inches by 4 inches.  The walls are 1 inch thick.  What is the total inner volume of all 3 boxes?

Rephrase 1: What is the combined internal volume of the three boxes that John has, considering each box measures 5 inches by 6 inches by 4 inches with 1-inch thick walls?
Rephrase 2: If John owns three boxes, and each box has dimensions of 5 inches by 6 inches by 4 inches with 1-inch thick walls, what is the total inner volume of these boxes?
Rephrase 3:  How much space is there inside all three of John's boxes when you account for their dimensions of 5 inches by 6 inches by 4 inches with walls that are 1 inch thick?

Question: Marcell and Beatrice are having a contest to see who can eat the most fruit roll-ups, so they unroll as many as they can find. Unfortunately, someone makes a mistake and Beatrice's was two roll-ups wide and 24 rolls up long while Marcell's was 3 roll-ups wide and 14 roll-ups long. If they both ate their entire amount, how many did they eat on average?
Rephrase 1: Marcell and Beatrice decided to have a competition to see who could consume the most fruit roll-ups, unrolling as many as they could find. However, there was a mistake, and Beatrice's roll-ups were two roll-ups wide and 24 roll-ups long, while Marcell's were three roll-ups wide and 14 roll-ups long. If they both ate their entire portions, what would be their average consumption?
Rephrase 2: In their contest to eat as many fruit roll-ups as possible, Marcell and Beatrice encountered a mix-up. Beatrice had roll-ups that were two roll-ups wide and 24 roll-ups long, and Marcell's were three roll-ups wide and 14 roll-ups long. If they finished eating all of their respective portions, what would be the average number of roll-ups consumed by each?
Rephrase 3: Marcell and Beatrice engaged in a fruit roll-up eating contest, but there was a mistake with the roll-ups. Beatrice had roll-ups that were two roll-ups wide and 24 roll-ups long, while Marcell's were three roll-ups wide and 14 roll-ups long. If they both ate their entire portions, what would be the average number of roll-ups they consumed?
"""

CLARIFY_PROMPT_AMBIGQA = """In what follows, you will be given some questions that might be ambiguous. These ambiguities can arise from various factors, including but not limited to:

1. Ambiguous references to entities in the question.
2. Multiple properties of objects/entities in the question leading to different interpretations.
3. Ambiguities due to unclear timestamps.
4. Ambiguities stemming from unclear locations.
5. Multiple valid answer types based on the question.

For each question, you are to provide at least two distinct rephrasings that resolve these ambiguities. By "rephrasing," we mean you should reformulate the question to be clear and direct, eliminating any possible ambiguity without altering the original intent of the question. You should not seek further information or produce a binary (yes-no) question as a result of the clarification. Instead, you must create a direct question (wh-question) that aims to obtain a specific answer.

Please format your responses as follows (with at least two rephrasings per question):
Clarifications:
1. [First rephrased question]
2. [Second rephrased question]
3. [Third rephrased question]
...

If the original question is already clear and unambiguous, you should indicate this by stating, "No clarification needed."
"""


def extract_clarifications(text, dataset_name):
    results = []
    lines = text.strip().split('\n')
    if dataset_name in ('nq_open', 'gsm8k'):
        for line in lines:
            if line.strip().startswith('Rephrase'):
                parts = line.split(':', 1)
                if len(parts) == 2:
                    results.append(parts[1].strip())
    elif dataset_name == 'ambigqa':
        for line in lines:
            if line.strip().startswith('Clarifications'):
                continue
            m = re.match(r'^\d+\.\s*(.+)', line.strip())
            if m:
                results.append(m.group(1))
    return results


def generate_clarifications(data, dataset_name, n_clarify=5):
    if dataset_name == 'nq_open':
        sys_prompt = CLARIFY_PROMPT_NQ
    elif dataset_name == 'gsm8k':
        sys_prompt = CLARIFY_PROMPT_GSM8K
    elif dataset_name == 'ambigqa':
        sys_prompt = CLARIFY_PROMPT_AMBIGQA
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    results = []
    for item in tqdm(data, desc=f"Clarify ({dataset_name})"):
        q = item['question']
        if dataset_name in ('nq_open', 'gsm8k'):
            user_prompt = f"Question: {q}"
        elif dataset_name == 'ambigqa':
            user_prompt = f"Original Question: {q}"

        msgs = format_messages(sys_prompt, user_prompt)
        replies = chat_complete(MODEL_CLARIFY, msgs, temperature=1.0,
                                max_tokens=512, n=1)
        clarifs = extract_clarifications(replies[0], dataset_name)
        if not clarifs:
            clarifs = [q]
        clarifs = list(dict.fromkeys(clarifs))[:n_clarify]

        entry = copy.deepcopy(item)
        entry['self_clarification'] = clarifs
        results.append(entry)
    return results


## Step 2 — Forward Pass (Answer Sampling)

For each clarification, sample N=10 answers from the LLM using temperature=0.5.
This creates a distribution over answers for each clarified question.

- **NQ / AmbigQA**: Use few-shot QA prompt + clarified question
- **GSM8K**: Use few-shot CoT prompt + original question (clarifications
  determine the number of independent forward passes)


In [ ]:
# ── Few-shot prompts ──

FEWSHOT_NQ = """Q: Who sang who wants to be a millionaire in high society?
A: Frank Sinatra

Q: when did india participate in olympics for first time
A: 1900

Q: new amsterdam was the main settlement in the dutch colony of
A: New Netherland

Q: where is the new tottenham stadium being built
A: the London Borough of Haringey

Q: What animal is mr. big in the movie zootopia
A: Arctic shrew"""

FEWSHOT_AMBIGQA = """Q: Who sang who wants to be a millionaire in high society?
A: Frank Sinatra

Q: when did india participate in olympics for first time
A: 1900

Q: Who played Bob in the film Elvira: Mistress of the Dark (2001)?
A: Invalid question. there was no 2001 version of the film "Elvira: Mistress of the Dark

Q: new amsterdam was the main settlement in the dutch colony of
A: New Netherland

Q: where is the new tottenham stadium being built
A: the London Borough of Haringey

Q: What animal is mr. big in the movie zootopia
A: Arctic shrew"""

FEWSHOT_GSM8K = """Question: Bella has two times as many marbles as frisbees. She also has 20 more frisbees than deck cards. If she buys 2/5 times more of each item, what would be the total number of the items she will have if she currently has 60 marbles?
Let's think step by step
When Bella buys 2/5 times more marbles, she'll have increased the number of marbles by 2/5*60 = 24
The total number of marbles she'll have is 60+24 = 84
If Bella currently has 60 marbles, and she has two times as many marbles as frisbees, she has 60/2 = 30 frisbees.
If Bella buys 2/5 times more frisbees, she'll have 2/5*30 = 12 more frisbees.
The total number of frisbees she'll have will increase to 30+12 = 42
Bella also has 20 more frisbees than deck cards, meaning she has 30-20 = 10 deck cards
If she buys 2/5 times more deck cards, she'll have 2/5*10 = 4 more deck cards.
The total number of deck cards she'll have is 10+4 = 14
Together, Bella will have a total of 14+42+84 = 140 items
The answer is 140

Question: A group of 4 fruit baskets contains 9 apples, 15 oranges, and 14 bananas in the first three baskets and 2 less of each fruit in the fourth basket. How many fruits are there?
Let's think step by step
For the first three baskets, the number of apples and oranges in one basket is 9+15=24
In total, together with bananas, the number of fruits in one basket is 24+14=38 for the first three baskets.
Since there are three baskets each having 38 fruits, there are 3*38=114 fruits in the first three baskets.
The number of apples in the fourth basket is 9-2=7
There are also 15-2=13 oranges in the fourth basket
The combined number of oranges and apples in the fourth basket is 13+7=20
The fourth basket also contains 14-2=12 bananas.
In total, the fourth basket has 20+12=32 fruits.
The four baskets together have 32+114=146 fruits.
The answer is 146"""


def forward_pass(data, dataset_name):
    if dataset_name in ('nq_open', 'ambigqa'):
        fewshot = FEWSHOT_NQ if dataset_name == 'nq_open' else FEWSHOT_AMBIGQA
        sys_prompt = ""
    elif dataset_name == 'gsm8k':
        fewshot = FEWSHOT_GSM8K
        sys_prompt = "Follow the given examples and answer the question."

    results = []
    for item in tqdm(data, desc=f"Forward ({dataset_name})"):
        entry = copy.deepcopy(item)
        clarifs = item.get('self_clarification', [item['question']])
        if not clarifs:
            clarifs = [item['question']]
        clarifs = clarifs[:MAX_CLARIFY]

        all_answers = []
        for cq in clarifs:
            if dataset_name in ('nq_open', 'ambigqa'):
                user_prompt = fewshot.strip() + "\n\nQ: " + cq
            elif dataset_name == 'gsm8k':
                user_prompt = fewshot.strip() + "\n\nQuestion: " + item['question'] + "\n"

            msgs = format_messages(sys_prompt if sys_prompt else None, user_prompt)
            replies = chat_complete(MODEL_FORWARD, msgs,
                                    temperature=TEMPERATURE,
                                    max_tokens=512, n=SAMPLE_N)
            if dataset_name in ('nq_open', 'ambigqa'):
                processed = []
                for r in replies:
                    first_line = r.strip().split('\n')[0] if dataset_name == 'ambigqa' else r.strip()
                    processed.append(first_line)
                all_answers.append(processed)
            elif dataset_name == 'gsm8k':
                all_answers.append(replies)

        entry['clarified_all_ans'] = all_answers
        results.append(entry)
    return results


## Step 3 — Answer Extraction

Normalize raw LLM answers into short, comparable forms.

- **NQ / AmbigQA**: Use an LLM to extract short answers and cluster semantically
  equivalent ones (following Hou et al.'s answer extraction prompt)
- **GSM8K**: Extract the final numeric answer with regex (no LLM needed)


In [ ]:
EXTRACTION_PROMPT = """**Task: Answer Extraction from Sentences**

In this task, you will receive both a question and multiple sentences. Each sentence contains an answer to the question. Your primary goal is to extract a concise answer, which can be a single word or a short phrase, from each sentence. Again, ensure you only extract a short answer! If a short answer cannot be directly extracted, then summarize the whole sentence into a single word or a short phrase.

Additionally, while extracting answers, your secondary goal is to create an "answer set" that contains all distinct answers from previous questions. If the extracted answer has not appeared in the answer set, add it to the answer set.

**Important Rules**
1. If there is an answer in the answer set that is semantically equivalent to the extracted answer, use the answer from the answer set as the result. Do not introduce a new, slightly different answer.
2. Separate different answers in the answer set using "|".
3. Also, extract the answer as "Unknown" for the following cases:
    - The sentence claims that there is no answer to the question
    - The sentence claims it lacks sufficient information to answer the question
    - The sentence claims it depends on various factors and the answer cannot be determined

**Output Format**
Extraction 1/N: [extraction from 1st sentence]
Updated answer set: [ ]
Extraction 2/N: [extraction from 2nd sentence]
Updated answer set: [ ]
...
Final answer set: [ ]
"""


def extract_answers_qa(data, dataset_name):
    results = []
    for item in tqdm(data, desc=f"Extract ({dataset_name})"):
        entry = copy.deepcopy(item)
        all_extracted = []
        prev_ans_set = "[]"

        for clarify_idx, answers in enumerate(item['clarified_all_ans']):
            q = item['question']
            prompt_q = f"Q: {q}"

            ans2id = {}
            cleaned = []
            for a in answers:
                a_clean = a[len("A: "):] if a.startswith("A: ") else a
                cleaned.append(a_clean)
                if a_clean not in ans2id:
                    ans2id[a_clean] = len(ans2id)

            for aid, ca in enumerate(ans2id.keys()):
                prompt_q += f"\nA{aid+1}: {ca}"
            prompt_q += f"\nAnswer set at the begining: {prev_ans_set}"

            msgs = format_messages(EXTRACTION_PROMPT, prompt_q)
            reply = chat_complete(MODEL_EXTRACT, msgs, temperature=0,
                                  max_tokens=1500, n=1)[0]

            ext_lines = []
            ans_sets = []
            for line in reply.split('\n'):
                if line.strip().startswith("Extraction"):
                    parts = line.strip().split(":", 1)
                    if len(parts) == 2:
                        ext_lines.append(parts[1].strip())
                else:
                    ans_sets.append(line.strip())

            if len(ext_lines) == len(ans2id):
                mapped = [ext_lines[ans2id[a]] for a in cleaned]
                all_extracted.append(mapped)
                for s in reversed(ans_sets):
                    if s:
                        if s.startswith("Final answer set:"):
                            prev_ans_set = s[len("Final answer set:"):].strip()
                        elif s.startswith("Updated answer set"):
                            prev_ans_set = s.split(":", 1)[-1].strip()
                        else:
                            prev_ans_set = s
                        break
            else:
                all_extracted.append(cleaned)

        entry['ext_clarified_all_ans'] = all_extracted
        results.append(entry)
    return results


def extract_answers_gsm8k(data):
    results = []
    for item in data:
        entry = copy.deepcopy(item)
        all_extracted = []
        for answers in item['clarified_all_ans']:
            extracted = [str(gsm8k_extract_ans(a)) for a in answers]
            all_extracted.append(extracted)
        entry['ext_clarified_all_ans'] = all_extracted
        results.append(entry)
    return results


## Step 4 — Uncertainty Quantification

Compute the ICE entropy decomposition for each question:

$$H_{\text{total}} = \underbrace{\bar{H}_{\text{cond}}}_{\text{model unc.}} + \underbrace{\text{MI}}_{\text{data unc.}}$$

- Build a frequency matrix: one row per clarification, columns = unique answers
- **Model uncertainty** = average conditional entropy (mean of per-row entropies)
- **Data uncertainty** = MI = total entropy − model uncertainty


In [ ]:
def compute_entropy(vec):
    vec = vec + 1e-10
    vec = vec / np.sum(vec)
    return -np.sum(vec * np.log2(vec))


def uncertainty_quantification(data, dataset_name):
    results = []
    answer_key = 'ext_clarified_all_ans'

    for item in data:
        raw_sets = item[answer_key]
        raw_sets = [check_answers(s) for s in raw_sets]
        raw_sets = recursive_normalize(raw_sets)

        unique_labels = []
        for s in raw_sets:
            labels = sorted(s, key=len, reverse=True)
            for x in labels:
                if not any(x in ex for ex in unique_labels):
                    unique_labels.append(x)

        ans2idx = {a: i for i, a in enumerate(unique_labels)}
        idx2ans = {i: a for a, i in ans2idx.items()}

        num_rewrite = len(raw_sets)
        if num_rewrite == 0:
            results.append({
                'question': item['question'],
                'answer': item['answer'],
                'total_uncertainty': 1.0,
                'model_uncertainty': 0.0,
                'data_uncertainty': 1.0,
                'most_freq_ans': 'unknown',
                'label': item.get('label'),
            })
            continue

        fuzz = FuzzingDict(ans2idx)
        output_space = len(idx2ans)
        freq_mat = []
        mv_answers = []
        all_unk = False

        for ridx in range(num_rewrite):
            ans_list = raw_sets[ridx]
            if ans_list is None:
                all_unk = True
                break

            freq = np.zeros(output_space)
            if output_space <= 1 and 'unknown' in ans_list:
                continue

            for a in ans_list:
                if a == 'unknown':
                    freq += 1 / (output_space - 1)
                    freq[ans2idx['unknown']] -= 1 / (output_space - 1)
                else:
                    freq[fuzz(a)] += 1

            freq = freq / SAMPLE_N
            freq_mat.append(freq)
            mv_ans = majority_vote(ans_list)[0]
            mv_answers.append(mv_ans)

        if all_unk or len(freq_mat) == 0:
            results.append({
                'question': item['question'],
                'answer': item['answer'],
                'total_uncertainty': 1.0,
                'model_uncertainty': 1.0,
                'data_uncertainty': 0.0,
                'most_freq_ans': 'unknown',
                'label': item.get('label'),
            })
            continue

        freq_mat = np.stack(freq_mat, axis=0)
        cond_entropies = [compute_entropy(freq_mat[i]) for i in range(freq_mat.shape[0])]
        cond_entropies = np.array(cond_entropies)

        pred_posterior = np.mean(freq_mat, axis=0)
        total_entropy = compute_entropy(pred_posterior)

        most_freq_id = np.argmax(pred_posterior)
        most_freq_ans = fuzz.id2ans[most_freq_id]

        model_unc = np.mean(cond_entropies)
        data_unc = total_entropy - model_unc

        results.append({
            'question': item['question'],
            'answer': item['answer'],
            'total_uncertainty': float(total_entropy),
            'model_uncertainty': float(model_unc),
            'data_uncertainty': float(data_unc),
            'most_freq_ans': most_freq_ans,
            'label': item.get('label'),
        })

    return results


## Step 5 — Evaluation

**Mistake detection** (NQ, GSM8K):
- A prediction is "correct" if the LLM's majority-vote answer matches ground truth
- AUROC: can total uncertainty separate correct from incorrect predictions?

**Ambiguity detection** (AmbigQA):
- Each question has a label: ambiguous or unambiguous
- AUROC: can data uncertainty (MI) separate ambiguous from unambiguous questions?

For NQ, we use LLM-based scoring since exact match is too strict for free-text QA.


In [ ]:
NQ_EVAL_PROMPT = """Rate the level of consistency between the answer to the question and the reference answer, from 0 to 100.
Question: who makes up the state council in russia
Reference: governors and presidents
Answer: governors and presidents
Rating: 100

Question: when does real time with bill maher come back
Reference: November 9, 2018
Answer: September 8, 2000
Rating: 0"""


def evaluate_nq_correctness(uq_results):
    """Use LLM to score answer correctness for NQ."""
    for item in tqdm(uq_results, desc="NQ scoring"):
        q = item['question']
        gt = item['answer'] if isinstance(item['answer'], str) else ', '.join(item['answer'])
        pred = item['most_freq_ans']
        user_prompt = (f"Question: {q}\nReference: {gt}\n"
                       f"Answer: {pred}\nRating:")
        msgs = format_messages(NQ_EVAL_PROMPT, user_prompt)
        reply = chat_complete(MODEL_EVAL, msgs, temperature=0, max_tokens=10, n=1)[0]
        try:
            score = int(re.findall(r'\d+', reply)[0])
        except (IndexError, ValueError):
            score = 0
        item['gptscore'] = score
        item['correct'] = score > 70
    return uq_results


def compute_metrics_mistake(uq_results, dataset_name):
    """Mistake detection AUROC and best F1 using total uncertainty."""
    labels = []
    uncertainties = []
    corr_unc, wrong_unc = [], []

    for item in uq_results:
        if dataset_name == 'nq_open':
            correct = item.get('correct', False)
        elif dataset_name == 'gsm8k':
            gt = gsm8k_extract_ans(item['answer'])
            pred = item['most_freq_ans']
            try:
                pred_num = int(pred)
            except (ValueError, TypeError):
                pred_num = -1
            correct = (pred_num == gt)
        else:
            raise ValueError

        labels.append(correct)
        u = item['total_uncertainty']
        uncertainties.append(u)
        (corr_unc if correct else wrong_unc).append(u)

    labels = np.array(labels)
    xs = np.array(uncertainties)

    acc = np.mean(labels)
    auroc = roc_auc_score(labels, -xs) if len(set(labels)) > 1 else float('nan')

    best_f1 = 0.0
    for t in np.arange(1, 200) / 100:
        pred_mistake = xs >= t
        true_mistake = ~labels
        f1 = f1_score(true_mistake, pred_mistake, zero_division=0)
        best_f1 = max(best_f1, f1)

    return {
        'dataset': dataset_name,
        'task': 'Mistake Detection',
        'accuracy': float(acc),
        'auroc': float(auroc),
        'best_f1': float(best_f1),
        'mean_unc_correct': float(np.mean(corr_unc)) if corr_unc else 0,
        'mean_unc_wrong': float(np.mean(wrong_unc)) if wrong_unc else 0,
    }


def compute_metrics_ambiguity(uq_results):
    """Ambiguity detection AUROC and best F1 using data uncertainty."""
    labels = []
    uncertainties = []
    ambig_unc, unambig_unc = [], []

    for item in uq_results:
        label_list = item.get('label', [])
        is_ambig = 'singleAnswer' not in label_list
        labels.append(is_ambig)
        u = item['data_uncertainty']
        uncertainties.append(u)
        (ambig_unc if is_ambig else unambig_unc).append(u)

    labels = np.array(labels)
    xs = np.array(uncertainties)

    auroc = roc_auc_score(labels, xs) if len(set(labels)) > 1 else float('nan')

    best_f1 = 0.0
    for t in np.arange(1, 100) / 100:
        pred_ambig = xs > t
        f1 = f1_score(labels, pred_ambig, zero_division=0)
        best_f1 = max(best_f1, f1)

    return {
        'dataset': 'ambigqa',
        'task': 'Ambiguity Detection',
        'auroc': float(auroc),
        'best_f1': float(best_f1),
        'mean_unc_ambig': float(np.mean(ambig_unc)) if ambig_unc else 0,
        'mean_unc_unambig': float(np.mean(unambig_unc)) if unambig_unc else 0,
    }


## Full Pipeline

Run the complete 5-step ICE pipeline for each dataset.
Intermediate results are cached to `CACHE_DIR` after each step.


In [ ]:
def save_cache(data, name):
    path = os.path.join(CACHE_DIR, f"{name}.json")
    with open(path, 'w') as f:
        json.dump(data, f, indent=2, default=str)
    print(f"  cached -> {path}")

def load_cache(name):
    path = os.path.join(CACHE_DIR, f"{name}.json")
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return None


def run_pipeline(dataset_name, data, skip_extraction_llm=False):
    """Run the full ICE pipeline for one dataset."""
    tag = dataset_name

    # Step 1: Clarifications
    cached = load_cache(f"{tag}_step1_clarify")
    if cached and not RUN_PIPELINE:
        step1 = cached
        print(f"[{tag}] Step 1: loaded from cache ({len(step1)} items)")
    elif RUN_PIPELINE:
        print(f"[{tag}] Step 1: generating clarifications...")
        step1 = generate_clarifications(data, dataset_name, n_clarify=MAX_CLARIFY)
        save_cache(step1, f"{tag}_step1_clarify")
    else:
        raise RuntimeError(f"No cache for {tag} step 1 and RUN_PIPELINE=False")

    # Step 2: Forward pass
    cached = load_cache(f"{tag}_step2_forward")
    if cached and not RUN_PIPELINE:
        step2 = cached
        print(f"[{tag}] Step 2: loaded from cache ({len(step2)} items)")
    elif RUN_PIPELINE:
        print(f"[{tag}] Step 2: sampling answers...")
        step2 = forward_pass(step1, dataset_name)
        save_cache(step2, f"{tag}_step2_forward")
    else:
        raise RuntimeError(f"No cache for {tag} step 2 and RUN_PIPELINE=False")

    # Step 3: Answer extraction
    cached = load_cache(f"{tag}_step3_extract")
    if cached and not RUN_PIPELINE:
        step3 = cached
        print(f"[{tag}] Step 3: loaded from cache ({len(step3)} items)")
    elif RUN_PIPELINE:
        print(f"[{tag}] Step 3: extracting answers...")
        if dataset_name == 'gsm8k' or skip_extraction_llm:
            step3 = extract_answers_gsm8k(step2)
        else:
            step3 = extract_answers_qa(step2, dataset_name)
        save_cache(step3, f"{tag}_step3_extract")
    else:
        raise RuntimeError(f"No cache for {tag} step 3 and RUN_PIPELINE=False")

    # Step 4: Uncertainty quantification
    print(f"[{tag}] Step 4: computing uncertainty decomposition...")
    step4 = uncertainty_quantification(step3, dataset_name)
    save_cache(step4, f"{tag}_step4_uq")

    # Step 5: Evaluation
    print(f"[{tag}] Step 5: computing metrics...")
    if dataset_name in ('nq_open',):
        if RUN_PIPELINE:
            step4 = evaluate_nq_correctness(step4)
            save_cache(step4, f"{tag}_step4_uq_scored")
        metrics = compute_metrics_mistake(step4, dataset_name)
    elif dataset_name == 'gsm8k':
        metrics = compute_metrics_mistake(step4, dataset_name)
    elif dataset_name == 'ambigqa':
        metrics = compute_metrics_ambiguity(step4)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    return step4, metrics


## Experiment 1 — Natural Questions (NQ)

**Task**: Mistake detection (can uncertainty identify wrong answers?)

| Metric | Description |
|--------|-------------|
| AUROC | Area under ROC: total uncertainty separating correct/incorrect |
| Best F1 | Best F1 score over all thresholds for mistake prediction |


In [ ]:
print("=" * 60)
print("Natural Questions (NQ)")
print("=" * 60)
nq_data = load_nq(N_NQ)
nq_uq, nq_metrics = run_pipeline('nq_open', nq_data)

print("\n--- NQ Results ---")
for k, v in nq_metrics.items():
    print(f"  {k:25s}: {v}")


## Experiment 2 — AmbigQA

**Task**: Ambiguity detection (can data uncertainty identify ambiguous questions?)

| Metric | Description |
|--------|-------------|
| AUROC | Area under ROC: data uncertainty (MI) separating ambig/unambig |
| Best F1 | Best F1 score for ambiguity prediction |


In [ ]:
print("=" * 60)
print("AmbigQA")
print("=" * 60)
ambigqa_data = load_ambigqa(N_AMBIGQA)
ambigqa_uq, ambigqa_metrics = run_pipeline('ambigqa', ambigqa_data)

print("\n--- AmbigQA Results ---")
for k, v in ambigqa_metrics.items():
    print(f"  {k:25s}: {v}")


## Experiment 3 — GSM8K

**Task**: Mistake detection for math reasoning

| Metric | Description |
|--------|-------------|
| AUROC | Area under ROC: total uncertainty separating correct/incorrect |
| Best F1 | Best F1 score for mistake prediction |


In [ ]:
print("=" * 60)
print("GSM8K")
print("=" * 60)
gsm8k_data = load_gsm8k(N_GSM8K)
gsm8k_uq, gsm8k_metrics = run_pipeline('gsm8k', gsm8k_data)

print("\n--- GSM8K Results ---")
for k, v in gsm8k_metrics.items():
    print(f"  {k:25s}: {v}")


## Results Summary

In [ ]:
all_metrics = [nq_metrics, ambigqa_metrics, gsm8k_metrics]

print("\n" + "=" * 70)
print(f"{'Dataset':<12} {'Task':<22} {'AUROC':>8} {'Best F1':>8} {'Acc':>8}")
print("-" * 70)
for m in all_metrics:
    ds = m['dataset']
    task = m['task']
    auroc = f"{m['auroc']:.3f}"
    f1 = f"{m['best_f1']:.3f}"
    acc = f"{m.get('accuracy', float('nan')):.3f}"
    print(f"{ds:<12} {task:<22} {auroc:>8} {f1:>8} {acc:>8}")
print("=" * 70)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── NQ: Correct vs Wrong uncertainty ──
ax = axes[0]
corr_u = [r['total_uncertainty'] for r in nq_uq if r.get('correct')]
wrong_u = [r['total_uncertainty'] for r in nq_uq if not r.get('correct')]
if corr_u and wrong_u:
    ax.hist(corr_u, bins=20, alpha=0.6, label=f'Correct (n={len(corr_u)})', color='#2ecc71')
    ax.hist(wrong_u, bins=20, alpha=0.6, label=f'Wrong (n={len(wrong_u)})', color='#e74c3c')
    ax.set_xlabel('Total Uncertainty')
    ax.set_ylabel('Count')
    ax.set_title(f"NQ — Mistake Detection\nAUROC={nq_metrics['auroc']:.3f}")
    ax.legend()

# ── AmbigQA: Ambiguous vs Unambiguous ──
ax = axes[1]
ambig_u = [r['data_uncertainty'] for r in ambigqa_uq
           if r.get('label') and 'singleAnswer' not in r['label']]
unambig_u = [r['data_uncertainty'] for r in ambigqa_uq
             if r.get('label') and 'singleAnswer' in r['label']]
if ambig_u and unambig_u:
    ax.hist(ambig_u, bins=20, alpha=0.6, label=f'Ambiguous (n={len(ambig_u)})', color='#e67e22')
    ax.hist(unambig_u, bins=20, alpha=0.6, label=f'Unambiguous (n={len(unambig_u)})', color='#3498db')
    ax.set_xlabel('Data Uncertainty (MI)')
    ax.set_ylabel('Count')
    ax.set_title(f"AmbigQA — Ambiguity Detection\nAUROC={ambigqa_metrics['auroc']:.3f}")
    ax.legend()

# ── GSM8K: Correct vs Wrong uncertainty ──
ax = axes[2]
corr_u_g, wrong_u_g = [], []
for r in gsm8k_uq:
    gt = gsm8k_extract_ans(r['answer'])
    try:
        pred_num = int(r['most_freq_ans'])
    except (ValueError, TypeError):
        pred_num = -1
    (corr_u_g if pred_num == gt else wrong_u_g).append(r['total_uncertainty'])
if corr_u_g and wrong_u_g:
    ax.hist(corr_u_g, bins=20, alpha=0.6, label=f'Correct (n={len(corr_u_g)})', color='#2ecc71')
    ax.hist(wrong_u_g, bins=20, alpha=0.6, label=f'Wrong (n={len(wrong_u_g)})', color='#e74c3c')
    ax.set_xlabel('Total Uncertainty')
    ax.set_ylabel('Count')
    ax.set_title(f"GSM8K — Mistake Detection\nAUROC={gsm8k_metrics['auroc']:.3f}")
    ax.legend()

plt.suptitle("ICE Uncertainty Decomposition — LLM Experiments (Hou et al. 2024)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("ice_llm_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved figure to ice_llm_results.png")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets_info = [
    ("NQ", nq_uq, 'total_uncertainty', 'model_uncertainty', 'data_uncertainty'),
    ("AmbigQA", ambigqa_uq, 'total_uncertainty', 'model_uncertainty', 'data_uncertainty'),
    ("GSM8K", gsm8k_uq, 'total_uncertainty', 'model_uncertainty', 'data_uncertainty'),
]

for ax, (name, uq_data, tk, mk, dk) in zip(axes, datasets_info):
    total = [r[tk] for r in uq_data]
    model = [r[mk] for r in uq_data]
    data_ = [r[dk] for r in uq_data]

    x = np.arange(3)
    means = [np.mean(total), np.mean(model), np.mean(data_)]
    stds = [np.std(total), np.std(model), np.std(data_)]
    colors = ['#34495e', '#9b59b6', '#1abc9c']
    bars = ax.bar(x, means, yerr=stds, color=colors, alpha=0.8, capsize=5)
    ax.set_xticks(x)
    ax.set_xticklabels(['Total H', 'Model Unc\n(avg cond H)', 'Data Unc\n(MI)'])
    ax.set_ylabel('Entropy (bits)')
    ax.set_title(f'{name}')

    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{m:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle("ICE Uncertainty Decomposition Breakdown", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("ice_llm_decomposition.png", dpi=150, bbox_inches='tight')
plt.show()
